In [ ]:
# Phase 0 — Cell 1: Install Required Libraries
!pip install faiss-cpu sentence-transformers langchain langgraph PyPDF2 pandas numpy requests

In [ ]:
# Phase 0 — Cell 2: Imports
import os
import json
import csv
import random
import time
import datetime
import numpy as np
import pandas as pd
import faiss
import requests

from sentence_transformers import SentenceTransformer

In [ ]:
# Phase 0 — Cell 3: Fix Random Seeds & Global Config

# --- Seeds ---
random.seed(42)
np.random.seed(42)

# --- Paths ---
LAW_TEXT_PATH      = "law_documents/pakistan_traffic_laws.txt"
DETECTION_LOG_PATH = "detection_logs.json"
FAISS_INDEX_PATH   = "rag/law_index.faiss"
METADATA_PATH      = "rag/law_metadata.csv"
CITATIONS_PATH     = "rag/citation_reports.json"

# --- Chunking ---
CHUNK_SIZE    = 300   # words
CHUNK_OVERLAP = 50    # words

# --- Retrieval ---
TOP_K = 5             # default; we'll also test 3 and 10

# --- Embedding ---
EMBEDDING_MODEL = "all-MiniLM-L6-v2"

# --- Ollama ---
OLLAMA_URL   = "http://localhost:11434/api/generate"
OLLAMA_MODEL = "llama3.2"

# --- Agent ---
CONFIDENCE_THRESHOLD     = 0.60   # Observer Agent filter
LOW_CONFIDENCE_THRESHOLD = 0.50   # Legal Consultant flag

print("✅ Phase 0 complete — all config set.")

In [ ]:
# Phase 1.2 — Cell 4: Load and Clean Raw Law Text

with open(LAW_TEXT_PATH, "r", encoding="utf-8") as f:
    raw_text = f.read()

# Basic cleaning
def clean_text(text):
    import re
    text = re.sub(r'\r\n', '\n', text)           # normalize line endings
    text = re.sub(r'\n{3,}', '\n\n', text)        # collapse excess blank lines
    text = re.sub(r'[ \t]+', ' ', text)           # collapse extra spaces/tabs
    text = text.strip()
    return text

clean_law_text = clean_text(raw_text)

print(f"✅ Law text loaded and cleaned.")
print(f"   Total characters : {len(clean_law_text)}")
print(f"   Total words      : {len(clean_law_text.split())}")
print(f"\n--- Preview (first 300 chars) ---\n{clean_law_text[:300]}")

In [ ]:
# Phase 1.2 — Cell 5: Section-Aware Sliding Window Chunker

import re

def extract_sections(text):
    """Split law text into (section_id, heading, body) tuples."""
    # Matches patterns like: 'Section 1.1 - Heading Title'
    pattern = re.compile(r'(Section\s[\d\.]+\s*[-–]\s*[^\n]+)', re.IGNORECASE)
    splits = pattern.split(text)

    sections = []
    # splits alternates: [preamble, heading, body, heading, body, ...]
    i = 1
    while i < len(splits) - 1:
        heading_raw = splits[i].strip()
        body        = splits[i + 1].strip()

        # Extract section ID (e.g. "1.1") from heading
        id_match   = re.search(r'[\d]+\.[\d]+', heading_raw)
        section_id = id_match.group(0) if id_match else f"sec_{i}"

        sections.append({
            "section_id" : section_id,
            "heading"    : heading_raw,
            "body"       : body
        })
        i += 2

    return sections


def sliding_window_chunks(section_id, heading, body, chunk_size=CHUNK_SIZE, overlap=CHUNK_OVERLAP):
    """
    Chunk a section body using a sliding window over words.
    chunk_size = 300 words, overlap = 50 words.
    
    Justification: 300-word chunks are large enough to preserve full legal
    context (penalty + rule + condition) while staying within the embedding
    model's 256-token sweet spot. 50-word overlap ensures no clause is split
    across two chunks without representation in both.
    """
    words  = body.split()
    chunks = []
    start  = 0
    idx    = 0

    while start < len(words):
        end        = min(start + chunk_size, len(words))
        chunk_text = " ".join(words[start:end])

        chunks.append({
            "chunk_id"   : f"{section_id}_chunk{idx}",
            "section_id" : section_id,
            "heading"    : heading,
            "text"       : chunk_text,
            "source"     : LAW_TEXT_PATH
        })

        if end == len(words):
            break

        start += chunk_size - overlap
        idx   += 1

    return chunks


# --- Run ---
sections   = extract_sections(clean_law_text)
all_chunks = []

for sec in sections:
    chunks = sliding_window_chunks(sec["section_id"], sec["heading"], sec["body"])
    all_chunks.extend(chunks)

print(f"✅ Chunking complete.")
print(f"   Sections found : {len(sections)}")
print(f"   Total chunks   : {len(all_chunks)}")

In [ ]:
# Phase 1.2 — Cell 6: Inspect Chunks

# Word count stats
word_counts = [len(c["text"].split()) for c in all_chunks]
print(f"Chunk word count — min: {min(word_counts)}, max: {max(word_counts)}, avg: {np.mean(word_counts):.1f}")

# Check all 7 violation categories are represented
VIOLATION_KEYWORDS = {
    "Lane Discipline"              : ["lane", "solid line", "overtaking", "straddling"],
    "Speed Limit"                  : ["speed", "speeding", "km/h"],
    "Traffic Signal"               : ["signal", "red light", "traffic light"],
    "Pedestrian Right-of-Way"      : ["pedestrian", "crosswalk", "zebra"],
    "Emergency Vehicle Obstruction": ["emergency", "ambulance", "fire brigade"],
    "Reckless Driving"             : ["reckless", "negligent", "dangerous driving"],
    "Vehicle Category Restrictions": ["heavy vehicle", "truck", "bus", "restricted zone"]
}

print("\n--- Violation Category Coverage ---")
for category, keywords in VIOLATION_KEYWORDS.items():
    matched = any(
        any(kw.lower() in c["text"].lower() for kw in keywords)
        for c in all_chunks
    )
    status = "✅" if matched else "❌"
    print(f"  {status} {category}")

# Preview first 2 chunks
print("\n--- Sample Chunks ---")
for c in all_chunks[:2]:
    print(f"\nChunk ID : {c['chunk_id']}")
    print(f"Section  : {c['section_id']} | {c['heading']}")
    print(f"Words    : {len(c['text'].split())}")
    print(f"Text     : {c['text'][:200]}...")

In [ ]:
# Phase 1.3 — Cell 7: Load SentenceTransformer Embedding Model

from sentence_transformers import SentenceTransformer

print(f"Loading embedding model: {EMBEDDING_MODEL} ...")
embedder = SentenceTransformer(EMBEDDING_MODEL)

print(f"✅ Model loaded.")
print(f"   Embedding dimension: {embedder.get_sentence_embedding_dimension()}")

In [ ]:
# Phase 1.3 — Cell 8: Generate Embeddings for All Chunks

start_time = time.time()

chunk_texts = [c["text"] for c in all_chunks]

embeddings = embedder.encode(
    chunk_texts,
    batch_size=32,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True   # L2-normalize for cosine similarity via dot product
)

elapsed = time.time() - start_time

print(f"\n✅ Embeddings generated.")
print(f"   Total chunks embedded : {len(embeddings)}")
print(f"   Embedding shape       : {embeddings.shape}")
print(f"   Total time            : {elapsed:.2f}s")
print(f"   Avg time per chunk    : {(elapsed / len(embeddings)) * 1000:.2f} ms")

In [ ]:
# Phase 1.3 — Cell 9: Save Embeddings to Disk

EMBEDDINGS_PATH = "rag/law_embeddings.npy"
np.save(EMBEDDINGS_PATH, embeddings)

# Verify
loaded = np.load(EMBEDDINGS_PATH)
assert loaded.shape == embeddings.shape, "Shape mismatch after save/load!"

print(f"✅ Embeddings saved and verified.")
print(f"   Path  : {EMBEDDINGS_PATH}")
print(f"   Shape : {loaded.shape}")
print(f"   dtype : {loaded.dtype}")

In [ ]:
# Phase 1.4 — Cell 10: Build FAISS IndexFlatL2

import faiss
import os

embedding_dim = embeddings.shape[1]  # 384

# IndexFlatL2 — exact nearest neighbour search, no training required
index = faiss.IndexFlatL2(embedding_dim)

# Since embeddings are L2-normalized, L2 distance ≈ cosine distance
index.add(embeddings)

print(f"✅ FAISS index built.")
print(f"   Index type       : IndexFlatL2")
print(f"   Embedding dim    : {embedding_dim}")
print(f"   Vectors indexed  : {index.ntotal}")

In [ ]:
# Phase 1.4 — Cell 11: Save FAISS Index and Metadata CSV

os.makedirs("rag", exist_ok=True)

# Save FAISS index
faiss.write_index(index, FAISS_INDEX_PATH)
print(f"✅ FAISS index saved to: {FAISS_INDEX_PATH}")

# Save metadata CSV
metadata_records = []
for i, chunk in enumerate(all_chunks):
    metadata_records.append({
        "chunk_index" : i,
        "chunk_id"    : chunk["chunk_id"],
        "section_id"  : chunk["section_id"],
        "heading"     : chunk["heading"],
        "text"        : chunk["text"],
        "source"      : chunk["source"]
    })

metadata_df = pd.DataFrame(metadata_records)
metadata_df.to_csv(METADATA_PATH, index=False, encoding="utf-8")
print(f"✅ Metadata saved to  : {METADATA_PATH}")
print(f"   Total records      : {len(metadata_df)}")

In [ ]:
# Phase 1.4 — Cell 12: Load Index Back and Verify

# Load
index_loaded   = faiss.read_index(FAISS_INDEX_PATH)
metadata_loaded = pd.read_csv(METADATA_PATH)

assert index_loaded.ntotal == len(metadata_loaded), "Index/metadata size mismatch!"

print(f"✅ Index and metadata loaded and verified.")
print(f"   Vectors in index : {index_loaded.ntotal}")
print(f"   Metadata rows    : {len(metadata_loaded)}")
print(f"\n--- Metadata Sample ---")
print(metadata_loaded[["chunk_id", "section_id", "heading"]].head(5).to_string(index=False))

In [ ]:
# Phase 1.5 — Cell 13: Retriever Helper Function

def retrieve_chunks(query, k=3, index=index_loaded, metadata=metadata_loaded):
    """
    Encode query → search FAISS → return top-k chunks with similarity scores.
    Since embeddings are L2-normalized, cosine_similarity = 1 - (L2_distance / 2)
    """
    query_embedding = embedder.encode(
        [query],
        convert_to_numpy=True,
        normalize_embeddings=True
    )

    distances, indices = index.search(query_embedding, k)

    results = []
    for rank, (dist, idx) in enumerate(zip(distances[0], indices[0])):
        cosine_sim = 1 - (dist / 2)   # convert L2 distance to cosine similarity
        row = metadata.iloc[idx]
        results.append({
            "rank"       : rank + 1,
            "chunk_id"   : row["chunk_id"],
            "section_id" : row["section_id"],
            "heading"    : row["heading"],
            "text"       : row["text"],
            "cosine_sim" : round(float(cosine_sim), 4)
        })

    return results

In [ ]:
# Phase 1.5 — Cell 14: Spot-Check — Query Index with 5 Violation Descriptions

spot_check_queries = [
    "Vehicle crossed a solid white line while overtaking",           # Lane Discipline
    "Car detected speeding above the legal speed limit on highway",  # Speed Limit
    "Vehicle ran a red traffic signal at an intersection",           # Traffic Signal
    "Pedestrian right of way was not given at a zebra crossing",     # Pedestrian
    "Heavy truck found driving in a restricted residential zone"     # Vehicle Category
]

spot_check_results = {}

print("=" * 70)
for query in spot_check_queries:
    print(f"\n Query: {query}")
    print("-" * 70)
    results = retrieve_chunks(query, k=3)
    spot_check_results[query] = results
    for r in results:
        print(f"  Rank {r['rank']} | Section {r['section_id']} | Sim: {r['cosine_sim']:.4f}")
        print(f"  Heading : {r['heading']}")
        print(f"  Preview : {r['text'][:150]}...")
        print()
print("=" * 70)

In [ ]:
# Phase 1.5 — Cell 15: Report Average Cosine Similarity of Top-1 Retrieved Chunk

top1_similarities = []

for query, results in spot_check_results.items():
    top1_sim = results[0]["cosine_sim"]
    top1_similarities.append(top1_sim)
    print(f"  Query : {query[:60]}...")
    print(f"  Top-1 Cosine Similarity : {top1_sim:.4f}\n")

avg_sim = np.mean(top1_similarities)
print(f"{'='*50}")
print(f"  Average Top-1 Cosine Similarity : {avg_sim:.4f}")
print(f"{'='*50}")

if avg_sim >= 0.5:
    print("✅ Retrieval quality acceptable (avg similarity >= 0.50)")
else:
    print("⚠️  Low average similarity — consider re-checking chunking or embeddings")

In [ ]:
# Phase 1.5 — Cell 16: Confirm All 7 Violation Categories in Knowledge Base

category_queries = {
    "Lane Discipline Violations"           : "vehicle crossing solid white line illegal lane change",
    "Speed Limit Violations"               : "vehicle exceeding speed limit on highway",
    "Traffic Signal Violations"            : "running red light at traffic signal intersection",
    "Pedestrian Right-of-Way Violations"   : "pedestrian crossing right of way zebra crossing",
    "Emergency Vehicle Obstruction"        : "blocking ambulance emergency vehicle on road",
    "Reckless or Negligent Driving"        : "reckless negligent dangerous driving behavior",
    "Vehicle Category Restrictions"        : "heavy truck bus restricted zone motorway lane"
}

print(f"{'Category':<45} {'Top-1 Section':<15} {'Similarity'}")
print("-" * 75)

all_covered = True
for category, query in category_queries.items():
    results = retrieve_chunks(query, k=1)
    top = results[0]
    covered = top["cosine_sim"] >= 0.30
    if not covered:
        all_covered = False
    status = "✅" if covered else "❌"
    print(f"{status} {category:<43} {top['section_id']:<15} {top['cosine_sim']:.4f}")

print()
if all_covered:
    print("✅ All 7 violation categories represented in the knowledge base.")
else:
    print("⚠️  Some categories may be missing — review law text coverage.")

In [ ]:
# Phase 2.1 — Cell 17: Query Encoder

def encode_query(query_text):
    """
    Encode a violation description string into a normalized embedding vector.
    Uses the same SentenceTransformer model as the knowledge base to ensure
    embedding space consistency.
    """
    start = time.time()

    query_embedding = embedder.encode(
        [query_text],
        convert_to_numpy=True,
        normalize_embeddings=True   # must match how chunks were embedded
    )

    latency_ms = (time.time() - start) * 1000

    return query_embedding, latency_ms


# --- Quick sanity check ---
test_query     = "Car ran a red light at a busy intersection"
test_embedding, latency = encode_query(test_query)

print(f"✅ Query Encoder ready.")
print(f"   Query            : {test_query}")
print(f"   Embedding shape  : {test_embedding.shape}")
print(f"   Embedding dtype  : {test_embedding.dtype}")
print(f"   Encoding latency : {latency:.2f} ms")
print(f"   L2 norm (≈1.0)   : {np.linalg.norm(test_embedding):.6f}")

In [ ]:
# Phase 2.2 — Cell 18: Retriever Function

def retrieve(query_text, k=TOP_K, index=index_loaded, metadata=metadata_loaded):
    """
    Full retriever:
      1. Encode query using Query Encoder
      2. Search FAISS index for top-k nearest chunks
      3. Return ranked results with cosine similarity scores
    """
    query_embedding, encode_latency = encode_query(query_text)

    faiss_start        = time.time()
    distances, indices = index.search(query_embedding, k)
    faiss_latency_ms   = (time.time() - faiss_start) * 1000

    results = []
    for rank, (dist, idx) in enumerate(zip(distances[0], indices[0])):
        cosine_sim = 1 - (dist / 2)
        row        = metadata.iloc[idx]
        results.append({
            "rank"          : rank + 1,
            "chunk_id"      : row["chunk_id"],
            "section_id"    : row["section_id"],
            "heading"       : row["heading"],
            "text"          : row["text"],
            "cosine_sim"    : round(float(cosine_sim), 4)
        })

    return results, encode_latency, faiss_latency_ms

In [ ]:
# Phase 2.2 — Cell 19: Test Retrieval at k=3, k=5, k=10

test_query = "Truck blocking an ambulance on the motorway"

k_values      = [3, 5, 10]
k_latencies   = {}

print(f"Query: {test_query}\n")
print("=" * 70)

for k in k_values:
    results, enc_lat, faiss_lat = retrieve(test_query, k=k)
    total_latency = enc_lat + faiss_lat
    k_latencies[k] = {
        "encode_ms"     : round(enc_lat, 2),
        "faiss_ms"      : round(faiss_lat, 2),
        "total_ms"      : round(total_latency, 2)
    }

    print(f"\n--- k={k} ---")
    for r in results:
        print(f"  Rank {r['rank']:>2} | Section {r['section_id']:<8} | "
              f"Sim: {r['cosine_sim']:.4f} | {r['heading'][:55]}")
    print(f"  ⏱ Encode: {enc_lat:.2f}ms | FAISS: {faiss_lat:.2f}ms | "
          f"Total: {total_latency:.2f}ms")

print("\n" + "=" * 70)

In [ ]:
# Phase 2.2 — Cell 20: Retrieval Depth Summary Table

print("\n--- Retrieval Depth Experiment — Latency Summary ---\n")
print(f"{'k':<6} {'Encode (ms)':<15} {'FAISS (ms)':<15} {'Total (ms)':<15}")
print("-" * 51)

for k, lat in k_latencies.items():
    print(f"{k:<6} {lat['encode_ms']:<15} {lat['faiss_ms']:<15} {lat['total_ms']:<15}")

print()
print("Observations:")
print("  k=3  — Most precise; may miss edge-case laws if top chunk is slightly off-topic.")
print("  k=5  — Balanced context; recommended default for prompt construction.")
print("  k=10 — Richest context; risks injecting loosely related law chunks into prompt.")

In [ ]:
# Phase 2.3 — Cell 21: Prompt Builder

def build_prompt(violation_description, retrieved_chunks, violation_id=None, 
                 timestamp=None, vehicle_class=None):
    """
    Construct a structured prompt for the LLM that includes:
      - Violation context
      - Retrieved law chunks as grounding context
      - Strict output format instructions
    """

    # Format retrieved chunks into numbered legal context block
    law_context = ""
    for i, chunk in enumerate(retrieved_chunks):
        law_context += f"""
[Law Chunk {i+1}]
Section    : {chunk['section_id']}
Heading    : {chunk['heading']}
Legal Text : {chunk['text']}
Similarity : {chunk['cosine_sim']}
"""

    prompt = f"""You are a Pakistani traffic law legal consultant AI. Your task is to analyze 
a traffic violation and generate a formal Citation Report grounded strictly in 
the provided law chunks. Do NOT invent law sections not present in the context.

===== VIOLATION DETAILS =====
Violation ID  : {violation_id if violation_id else "N/A"}
Timestamp     : {timestamp if timestamp else "N/A"}
Vehicle Class : {vehicle_class if vehicle_class else "N/A"}
Description   : {violation_description}

===== RETRIEVED LEGAL CONTEXT =====
{law_context}

===== INSTRUCTIONS =====
Using ONLY the law chunks provided above, generate a Citation Report in the 
EXACT format below. Do not add extra fields. Do not skip any field.
For 'Legal Text', write a concise summary of the applicable chunk — do NOT 
copy it verbatim. For 'Confidence', use the similarity score of the most 
relevant chunk.

===== OUTPUT FORMAT =====
CITATION REPORT
----------------
Violation ID      : 
Detected At       : 
Vehicle           : 
Violation Type    : 
Description       : 
Applicable Law    : 
Legal Text        : 
Recommended Action: 
Confidence        : 
"""
    return prompt

In [ ]:
# Phase 2.3 — Cell 22: Test Prompt Builder with a Sample Violation

sample_query       = "Car ran a red light at a busy intersection"
sample_results, _, _ = retrieve(sample_query, k=TOP_K)

sample_prompt = build_prompt(
    violation_description = sample_query,
    retrieved_chunks      = sample_results,
    violation_id          = "VIO-TEST-001",
    timestamp             = "2024-01-15T08:42:11Z",
    vehicle_class         = "car"
)

print("✅ Prompt built successfully.")
print(f"   Prompt length     : {len(sample_prompt)} characters")
print(f"   Prompt word count : {len(sample_prompt.split())} words")
print(f"\n{'='*70}")
print(sample_prompt)
print(f"{'='*70}")

In [ ]:
# Phase 2.4 — Cell 23: Generator — Call Ollama REST API

def generate_citation_report(prompt, model=OLLAMA_MODEL):
    """
    Send prompt to local Ollama instance and return generated Citation Report.
    Calls: POST http://localhost:11434/api/generate
    """
    payload = {
        "model"  : model,
        "prompt" : prompt,
        "stream" : False    # get full response at once
    }

    try:
        start    = time.time()
        response = requests.post(OLLAMA_URL, json=payload, timeout=120)
        latency  = time.time() - start

        if response.status_code != 200:
            return None, latency, f"HTTP Error {response.status_code}: {response.text}"

        data           = response.json()
        generated_text = data.get("response", "").strip()

        return generated_text, latency, None   # (report, latency, error)

    except requests.exceptions.ConnectionError:
        return None, 0, "❌ Cannot connect to Ollama. Is 'ollama serve' running?"
    except requests.exceptions.Timeout:
        return None, 0, "❌ Ollama request timed out after 120s."
    except Exception as e:
        return None, 0, f"❌ Unexpected error: {str(e)}"

In [ ]:
# Phase 2.4 — Cell 24: Parse Generated Citation Report into Structured Dict

REPORT_FIELDS = [
    "Violation ID",
    "Detected At",
    "Vehicle",
    "Violation Type",
    "Description",
    "Applicable Law",
    "Legal Text",
    "Recommended Action",
    "Confidence"
]

def parse_citation_report(report_text):
    """
    Parse the LLM-generated Citation Report text into a structured dictionary.
    Also checks completeness — all 9 fields must be present and non-empty.
    """
    parsed = {}

    for field in REPORT_FIELDS:
        # Match field name followed by colon and capture value
        pattern = rf"{re.escape(field)}\s*:\s*(.+)"
        match   = re.search(pattern, report_text, re.IGNORECASE)
        parsed[field] = match.group(1).strip() if match else ""

    # Completeness check
    missing  = [f for f in REPORT_FIELDS if not parsed.get(f)]
    complete = len(missing) == 0

    return parsed, complete, missing

In [ ]:
# Phase 2.4 — Cell 25: End-to-End Generation Test

print("🔄 Sending prompt to Ollama (mistral)...")
print(f"   URL   : {OLLAMA_URL}")
print(f"   Model : {OLLAMA_MODEL}\n")

report_text, latency, error = generate_citation_report(sample_prompt)

if error:
    print(error)
else:
    print(f"✅ Citation Report generated in {latency:.2f}s\n")
    print("=" * 70)
    print(report_text)
    print("=" * 70)

    # Parse and validate
    parsed, complete, missing = parse_citation_report(report_text)

    print(f"\n--- Completeness Check ---")
    print(f"   All 9 fields present : {'✅ Yes' if complete else '❌ No'}")
    if missing:
        print(f"   Missing fields       : {missing}")

    print(f"\n--- Parsed Fields ---")
    for field, value in parsed.items():
        status = "✅" if value else "❌"
        print(f"   {status} {field:<22}: {value[:80] if value else 'MISSING'}")

In [ ]:
# Phase 2.4 — Cell 26: Retrieval Depth Experiment — k=3, k=5, k=10

depth_query = "Car ran a red light at a busy intersection"

depth_results_table = []

print(f"Query: {depth_query}\n")
print("=" * 70)

for k in [3, 5, 10]:
    # Retrieve
    chunks, enc_lat, faiss_lat = retrieve(depth_query, k=k)

    # Build prompt & generate
    prompt       = build_prompt(
        violation_description = depth_query,
        retrieved_chunks      = chunks,
        violation_id          = f"VIO-DEPTH-K{k}",
        timestamp             = "2024-01-15T08:42:11Z",
        vehicle_class         = "car"
    )
    report, gen_latency, error = generate_citation_report(prompt)

    if error:
        print(f"k={k} — ERROR: {error}")
        continue

    parsed, complete, missing = parse_citation_report(report)
    total_latency = enc_lat + faiss_lat + (gen_latency * 1000)

    depth_results_table.append({
        "k"                  : k,
        "chunks_retrieved"   : k,
        "complete_report"    : complete,
        "missing_fields"     : missing,
        "top1_similarity"    : chunks[0]["cosine_sim"],
        "topk_similarity"    : chunks[-1]["cosine_sim"],
        "encode_ms"          : round(enc_lat, 2),
        "faiss_ms"           : round(faiss_lat, 2),
        "llm_latency_s"      : round(gen_latency, 2),
        "total_latency_ms"   : round(total_latency, 2)
    })

    print(f"\n--- k={k} ---")
    print(f"  Complete       : {'✅' if complete else '❌'}")
    print(f"  Top-1 Sim      : {chunks[0]['cosine_sim']:.4f}")
    print(f"  Top-k Sim      : {chunks[-1]['cosine_sim']:.4f}")
    print(f"  LLM Latency    : {gen_latency:.2f}s")
    print(f"  Total Latency  : {total_latency/1000:.2f}s")
    print(f"\n  Report Preview :\n{report[:400]}...")

print("\n" + "=" * 70)

# Summary table
depth_df = pd.DataFrame(depth_results_table)
print("\n--- Retrieval Depth Summary Table ---\n")
print(depth_df[[
    "k", "complete_report", "top1_similarity",
    "topk_similarity", "llm_latency_s", "total_latency_ms"
]].to_string(index=False))

In [ ]:
# Phase 3.1 — Cell 27: Violation Type Classifier

def classify_violation(detections):
    """
    Classify violation type based on detected object combinations.
    Returns (violation_type, description) or (None, None) if no violation.
    
    Violation Type Mapping:
    ┌─────────────────────────────────┬──────────────────────────────────────┐
    │ Detection Combination           │ Violation Type                       │
    ├─────────────────────────────────┼──────────────────────────────────────┤
    │ person + traffic light          │ Pedestrian Signal Violation          │
    │ car/truck + traffic light       │ Red Light Running                    │
    │ rider/motor/bike                │ Reckless Driving — Vulnerable User   │
    │ truck/bus                       │ Heavy Vehicle Restriction            │
    │ car (high bbox area)            │ Speed Limit Violation                │
    └─────────────────────────────────┴──────────────────────────────────────┘
    """
    labels = [d["class_label"].lower() for d in detections]
    
    has_person        = "person" in labels
    has_traffic_light = "traffic light" in labels
    has_car           = "car" in labels
    has_truck         = "truck" in labels
    has_bus           = "bus" in labels
    has_rider         = "rider" in labels
    has_motor         = "motor" in labels
    has_bike          = "bike" in labels

    # Rule-based violation classification (priority order)
    if has_person and has_traffic_light:
        return (
            "Pedestrian Signal Violation",
            "A pedestrian was detected at a traffic signal — possible jaywalking "
            "or failure to comply with pedestrian signal."
        )

    if (has_car or has_truck) and has_traffic_light:
        return (
            "Red Light Running",
            "A vehicle was detected in proximity to a traffic light — possible "
            "red light violation at an intersection."
        )

    if has_rider or has_motor or has_bike:
        return (
            "Reckless Driving — Vulnerable Road User",
            "A motorcycle or bicycle rider was detected — possible reckless "
            "or negligent riding behavior by a vulnerable road user."
        )

    if has_truck or has_bus:
        return (
            "Heavy Vehicle Restriction",
            "A heavy vehicle (truck or bus) was detected — possible operation "
            "in a restricted zone or violation of motorway lane restrictions."
        )

    if has_car:
        # Estimate speed proxy via bounding box area
        car_det = next(d for d in detections if d["class_label"].lower() == "car")
        bbox    = car_det["bbox"]
        area    = (bbox[2] - bbox[0]) * (bbox[3] - bbox[1])
        if area > 100000:
            return (
                "Speed Limit Violation",
                "A car was detected occupying a large frame area at high proximity — "
                "possible speed limit violation."
            )

    return (None, None)

In [ ]:
# Phase 3.1 — Cell 28: Observer Agent

def observer_agent(frame):
    """
    Observer Agent:
      - Input  : Raw detection frame from YOLO log
      - Process: Filter by confidence >= 0.6, classify violation
      - Output : Structured violation JSON or None if no violation
    """
    frame_id  = frame["frame_id"]
    timestamp = frame["timestamp"]

    # Step 1 — Filter detections by confidence threshold
    confident_detections = [
        d for d in frame["detections"]
        if d["confidence"] >= CONFIDENCE_THRESHOLD
    ]

    if not confident_detections:
        print(f"  [{frame_id}] ⚠️  No detections above confidence threshold {CONFIDENCE_THRESHOLD}")
        return None

    # Step 2 — Classify violation
    violation_type, description = classify_violation(confident_detections)

    if not violation_type:
        print(f"  [{frame_id}] ℹ️  No violation pattern matched.")
        return None

    # Step 3 — Estimate location from average bbox center
    centers = []
    for d in confident_detections:
        bbox = d["bbox"]
        cx   = (bbox[0] + bbox[2]) / 2
        cy   = (bbox[1] + bbox[3]) / 2
        centers.append((cx, cy))
    avg_cx = round(np.mean([c[0] for c in centers]), 1)
    avg_cy = round(np.mean([c[1] for c in centers]), 1)

    # Step 4 — Get highest confidence detection
    top_detection = max(confident_detections, key=lambda d: d["confidence"])

    # Step 5 — Build structured violation JSON
    violation_event = {
        "violation_id"     : f"VIO-{frame_id.upper()}",
        "vehicle_class"    : top_detection["class_label"],
        "violation_type"   : violation_type,
        "description"      : description,
        "timestamp"        : timestamp,
        "frame_id"         : frame_id,
        "location_estimate": f"frame_center({avg_cx}, {avg_cy})",
        "confidence"       : round(top_detection["confidence"], 4),
        "detections_used"  : [
            {"class": d["class_label"], "confidence": d["confidence"]}
            for d in confident_detections
        ]
    }

    return violation_event

In [ ]:
# Phase 3.1 — Cell 29: Run Observer Agent on All Detection Log Frames

with open(DETECTION_LOG_PATH, "r") as f:
    detection_log = json.load(f)

print("=" * 70)
print("OBSERVER AGENT — Processing Detection Log")
print("=" * 70)

violation_events = []

for frame in detection_log:
    print(f"\n📷 Frame: {frame['frame_id']} | {frame['timestamp']}")
    print(f"   Raw detections : {len(frame['detections'])}")

    event = observer_agent(frame)

    if event:
        violation_events.append(event)
        print(f"   ✅ Violation    : {event['violation_type']}")
        print(f"   Vehicle        : {event['vehicle_class']}")
        print(f"   Confidence     : {event['confidence']}")
        print(f"   Location       : {event['location_estimate']}")

print(f"\n{'='*70}")
print(f"✅ Observer Agent complete.")
print(f"   Frames processed  : {len(detection_log)}")
print(f"   Violations found  : {len(violation_events)}")
print(f"   Frames skipped    : {len(detection_log) - len(violation_events)}")

In [ ]:
# Phase 3.2 — Cell 30: Legal Consultant Agent

def legal_consultant_agent(violation_event, k=TOP_K):
    """
    Legal Consultant Agent:
      - Input  : Structured violation JSON from Observer Agent
      - Process: Retrieve relevant law chunks, build prompt, generate Citation Report
      - Output : Completed Citation Report dict with metadata
    """

    violation_id  = violation_event["violation_id"]
    vehicle_class = violation_event["vehicle_class"]
    timestamp     = violation_event["timestamp"]
    description   = violation_event["description"]
    viol_type     = violation_event["violation_type"]

    # Step 1 — Build enriched query for FAISS retrieval
    query = f"{viol_type} {description} vehicle {vehicle_class} Pakistan traffic law"

    # Step 2 — Retrieve relevant law chunks
    retrieved_chunks, enc_lat, faiss_lat = retrieve(query, k=k)
    top_similarity = retrieved_chunks[0]["cosine_sim"]

    # Step 3 — Check similarity threshold
    low_confidence = top_similarity < LOW_CONFIDENCE_THRESHOLD

    # Step 4 — Build prompt
    prompt = build_prompt(
        violation_description = f"{viol_type}: {description}",
        retrieved_chunks      = retrieved_chunks,
        violation_id          = violation_id,
        timestamp             = timestamp,
        vehicle_class         = vehicle_class
    )

    # Step 5 — Generate Citation Report via Ollama
    report_text, gen_latency, error = generate_citation_report(prompt)

    if error:
        return {
            "violation_id"   : violation_id,
            "status"         : "ERROR",
            "error"          : error,
            "report_text"    : None,
            "parsed_report"  : None,
            "low_confidence" : True,
            "top_similarity" : top_similarity,
            "latency"        : {
                "encode_ms"  : round(enc_lat, 2),
                "faiss_ms"   : round(faiss_lat, 2),
                "llm_s"      : 0
            }
        }

    # Step 6 — Parse and validate report
    parsed_report, complete, missing = parse_citation_report(report_text)

    # Step 7 — Validate report references a real law section
    real_section_referenced = any(
        str(chunk["section_id"]) in report_text
        for chunk in retrieved_chunks
    )

    # Step 8 — Build output
    result = {
        "violation_id"            : violation_id,
        "status"                  : "LOW_CONFIDENCE" if low_confidence else "OK",
        "low_confidence"          : low_confidence,
        "top_similarity"          : top_similarity,
        "real_section_referenced" : real_section_referenced,
        "complete"                : complete,
        "missing_fields"          : missing,
        "report_text"             : report_text,
        "parsed_report"           : parsed_report,
        "retrieved_chunks"        : [
            {
                "section_id" : c["section_id"],
                "heading"    : c["heading"],
                "cosine_sim" : c["cosine_sim"]
            }
            for c in retrieved_chunks
        ],
        "latency" : {
            "encode_ms"  : round(enc_lat, 2),
            "faiss_ms"   : round(faiss_lat, 2),
            "llm_s"      : round(gen_latency, 2),
            "total_s"    : round((enc_lat + faiss_lat) / 1000 + gen_latency, 2)
        }
    }

    return result

In [ ]:
# Phase 3.2 — Cell 31: Test Legal Consultant Agent on First Violation Event

if not violation_events:
    print("❌ No violation events found. Run Cell 29 first.")
else:
    test_event = violation_events[0]

    print("=" * 70)
    print("LEGAL CONSULTANT AGENT — Single Violation Test")
    print("=" * 70)
    print(f"\n📋 Input Violation Event:")
    print(f"   Violation ID   : {test_event['violation_id']}")
    print(f"   Vehicle        : {test_event['vehicle_class']}")
    print(f"   Violation Type : {test_event['violation_type']}")
    print(f"   Timestamp      : {test_event['timestamp']}")
    print(f"   Description    : {test_event['description']}")

    print(f"\n🔄 Running Legal Consultant Agent...")
    result = legal_consultant_agent(test_event)

    print(f"\n✅ Citation Report Generated")
    print(f"   Status                  : {result['status']}")
    print(f"   Top Similarity          : {result['top_similarity']:.4f}")
    print(f"   Real Section Referenced : {'✅' if result['real_section_referenced'] else '❌'}")
    print(f"   Report Complete         : {'✅' if result['complete'] else '❌'}")
    if result["missing_fields"]:
        print(f"   Missing Fields          : {result['missing_fields']}")
    print(f"   LLM Latency             : {result['latency']['llm_s']:.2f}s")
    print(f"   Total Latency           : {result['latency']['total_s']:.2f}s")

    if result["low_confidence"]:
        print(f"\n⚠️  LOW CONFIDENCE FLAG — Similarity {result['top_similarity']:.4f} < {LOW_CONFIDENCE_THRESHOLD}")
        print(f"   This report requires human review.")

    print(f"\n--- Generated Citation Report ---")
    print(result["report_text"])

In [ ]:
# Phase 3.2 — Cell 32: Run Legal Consultant Agent on All Violation Events

print("=" * 70)
print("LEGAL CONSULTANT AGENT — Processing All Violation Events")
print("=" * 70)

citation_reports = []

for i, event in enumerate(violation_events):
    print(f"\n[{i+1}/{len(violation_events)}] Processing {event['violation_id']}...")
    print(f"  Type : {event['violation_type']}")

    result = legal_consultant_agent(event)
    citation_reports.append(result)

    print(f"  Status     : {result['status']}")
    print(f"  Similarity : {result['top_similarity']:.4f}")
    print(f"  Complete   : {'✅' if result['complete'] else '❌'}")
    print(f"  Latency    : {result['latency']['total_s']:.2f}s")

    if result["low_confidence"]:
        print(f"  ⚠️  LOW CONFIDENCE — human review required")

print(f"\n{'='*70}")
print(f"✅ Legal Consultant Agent complete.")
print(f"   Total reports generated : {len(citation_reports)}")
print(f"   Complete reports        : {sum(1 for r in citation_reports if r['complete'])}")
print(f"   Low confidence flags    : {sum(1 for r in citation_reports if r['low_confidence'])}")
print(f"   Errors                  : {sum(1 for r in citation_reports if r['status'] == 'ERROR')}")
avg_latency = np.mean([r['latency']['total_s'] for r in citation_reports])
print(f"   Avg total latency       : {avg_latency:.2f}s")

In [ ]:
# Phase 3.3 — Cell 33: Inter-Agent Message Bus

class MessageBus:
    """
    Shared message bus for inter-agent communication.
    Each message contains:
      - sender       : observer | legal_consultant
      - message_type : violation_event | citation_report | 
                       clarification_request | low_confidence_flag
      - payload      : JSON-serializable dict
      - timestamp    : ISO 8601
    """

    def __init__(self):
        self.messages = []

    def send(self, sender, message_type, payload):
        message = {
            "sender"       : sender,
            "message_type" : message_type,
            "payload"      : payload,
            "timestamp"    : datetime.datetime.utcnow().isoformat() + "Z"
        }
        self.messages.append(message)
        return message

    def get_messages(self, sender=None, message_type=None):
        """Filter messages by sender or type."""
        filtered = self.messages
        if sender:
            filtered = [m for m in filtered if m["sender"] == sender]
        if message_type:
            filtered = [m for m in filtered if m["message_type"] == message_type]
        return filtered

    def get_last(self, message_type=None):
        """Get last message optionally filtered by type."""
        messages = self.get_messages(message_type=message_type)
        return messages[-1] if messages else None

    def log(self):
        """Pretty-print full message bus log."""
        print(f"\n{'='*70}")
        print(f"MESSAGE BUS LOG — {len(self.messages)} messages")
        print(f"{'='*70}")
        for i, msg in enumerate(self.messages):
            print(f"\n[MSG {i+1}]")
            print(f"  Sender       : {msg['sender']}")
            print(f"  Type         : {msg['message_type']}")
            print(f"  Timestamp    : {msg['timestamp']}")
            print(f"  Payload Keys : {list(msg['payload'].keys())}")
        print(f"\n{'='*70}")

    def export(self):
        """Return full log as list of dicts."""
        return self.messages


# Initialize shared bus
bus = MessageBus()
print("✅ Message Bus initialized.")

In [ ]:
# Phase 3.3 — Cell 34: Full Pipeline — Observer + Legal Consultant via Message Bus

def run_pipeline_cycle(frame, bus, k=TOP_K):
    """
    Run one complete violation → citation cycle through the message bus.
    
    Cycle:
      1. Observer Agent processes frame → sends violation_event
      2. Legal Consultant receives event → sends citation_report
      3. If low confidence → Legal Consultant sends low_confidence_flag
    """
    frame_id = frame["frame_id"]
    print(f"\n{'─'*70}")
    print(f"🔄 CYCLE START — {frame_id}")
    print(f"{'─'*70}")

    # ── Step 1: Observer Agent ──────────────────────────────────────────────
    print(f"\n[Observer Agent] Processing {frame_id}...")
    violation_event = observer_agent(frame)

    if not violation_event:
        # Send no-violation message
        bus.send(
            sender       = "observer",
            message_type = "violation_event",
            payload      = {
                "frame_id" : frame_id,
                "status"   : "NO_VIOLATION",
                "reason"   : "No violation pattern matched or low confidence detections"
            }
        )
        print(f"[Observer Agent] No violation detected — skipping cycle.")
        return None

    # Send violation event to bus
    bus.send(
        sender       = "observer",
        message_type = "violation_event",
        payload      = violation_event
    )
    print(f"[Observer Agent] ✅ Violation detected: {violation_event['violation_type']}")
    print(f"[Observer Agent] 📤 Sent violation_event → bus")

    # ── Step 2: Legal Consultant Agent ─────────────────────────────────────
    print(f"\n[Legal Consultant] Received violation_event from bus...")
    result = legal_consultant_agent(violation_event, k=k)

    if result["status"] == "ERROR":
        bus.send(
            sender       = "legal_consultant",
            message_type = "clarification_request",
            payload      = {
                "violation_id" : violation_event["violation_id"],
                "reason"       : result["error"]
            }
        )
        print(f"[Legal Consultant] ❌ Error — sent clarification_request → bus")
        return result

    # Send citation report to bus
    bus.send(
        sender       = "legal_consultant",
        message_type = "citation_report",
        payload      = {
            "violation_id"  : result["violation_id"],
            "status"        : result["status"],
            "parsed_report" : result["parsed_report"],
            "report_text"   : result["report_text"],
            "top_similarity": result["top_similarity"],
            "complete"      : result["complete"],
            "latency"       : result["latency"]
        }
    )
    print(f"[Legal Consultant] ✅ Citation Report generated")
    print(f"[Legal Consultant] 📤 Sent citation_report → bus")

    # ── Step 3: Low Confidence Flag ────────────────────────────────────────
    if result["low_confidence"]:
        bus.send(
            sender       = "legal_consultant",
            message_type = "low_confidence_flag",
            payload      = {
                "violation_id"  : result["violation_id"],
                "top_similarity": result["top_similarity"],
                "threshold"     : LOW_CONFIDENCE_THRESHOLD,
                "action"        : "Human review required"
            }
        )
        print(f"[Legal Consultant] ⚠️  LOW CONFIDENCE — 📤 Sent low_confidence_flag → bus")

    print(f"\n✅ CYCLE COMPLETE — {frame_id}")
    return result

In [ ]:
# Phase 3.3 — Cell 35: Run and Document 3 Complete Violation → Citation Cycles

# Reset bus for clean 3-cycle log
bus = MessageBus()

# Pick 3 frames that produced violation events
cycle_frames  = [
    frame for frame in detection_log
    if any(e["frame_id"] == frame["frame_id"] for e in violation_events)
][:3]

print("=" * 70)
print("DUAL-AGENT PIPELINE — 3 Complete Cycles")
print("=" * 70)

cycle_results = []

for i, frame in enumerate(cycle_frames):
    print(f"\n{'#'*70}")
    print(f"# CYCLE {i+1} of 3")
    print(f"{'#'*70}")

    result = run_pipeline_cycle(frame, bus)

    if result and result.get("report_text"):
        cycle_results.append(result)
        print(f"\n--- Citation Report (Cycle {i+1}) ---")
        print(result["report_text"])

print(f"\n{'='*70}")
print(f"✅ 3 Cycles Complete")
print(f"   Successful reports  : {len(cycle_results)}")
print(f"   Total bus messages  : {len(bus.messages)}")

In [ ]:
# Phase 3.3 — Cell 36: Print Full Inter-Agent Message Bus Log

bus.log()

print("\n--- Detailed Message Payloads ---")
for i, msg in enumerate(bus.messages):
    print(f"\n[MSG {i+1}] {msg['sender'].upper()} → {msg['message_type']}")
    print(f"  Timestamp : {msg['timestamp']}")
    payload = msg["payload"]
    for key, val in payload.items():
        if key in ["report_text", "parsed_report"]:
            # Truncate long fields
            preview = str(val)[:120] + "..." if val and len(str(val)) > 120 else str(val)
            print(f"  {key:<20}: {preview}")
        else:
            print(f"  {key:<20}: {val}")

In [ ]:
# Phase 3.3 — Cell 37: Export Message Bus Log to JSON

MESSAGE_LOG_PATH = "rag/message_bus_log.json"

with open(MESSAGE_LOG_PATH, "w", encoding="utf-8") as f:
    json.dump(bus.export(), f, indent=2, default=str)

print(f"✅ Message bus log exported to: {MESSAGE_LOG_PATH}")
print(f"   Total messages logged: {len(bus.messages)}")

# Summary by message type
from collections import Counter
type_counts = Counter(m["message_type"] for m in bus.messages)
print(f"\n--- Message Type Summary ---")
for mtype, count in type_counts.items():
    print(f"   {mtype:<25}: {count}")

In [ ]:
# Phase 4.1 — Cell 38: Ground Truth Set (20 Violation Events)

ground_truth = [
    # Format: (violation_description, correct_section_id, violation_type)
    ("Vehicle crossed a solid white line while overtaking",                    "1.1", "Lane Discipline Violations"),
    ("Driver changed lanes abruptly without signaling on highway",             "1.2", "Lane Discipline Violations"),
    ("Car overtook another vehicle at a pedestrian crossing",                  "1.3", "Lane Discipline Violations"),
    ("Vehicle failed to comply with road lane markings",                       "1.4", "Lane Discipline Violations"),
    ("Car drove against traffic flow on a one-way road",                       "1.5", "Lane Discipline Violations"),
    ("Vehicle exceeded speed limit on motorway",                               "2.1", "Speed Limit Violations"),
    ("Car detected driving at 130 km/h in an 80 km/h zone",                   "2.2", "Speed Limit Violations"),
    ("Vehicle ran a red traffic signal at an intersection",                    "3.1", "Traffic Signal Violations"),
    ("Driver accelerated through an amber light at intersection",              "3.3", "Traffic Signal Violations"),
    ("Vehicle stopped beyond the stop line at a red signal",                   "3.4", "Traffic Signal Violations"),
    ("Pedestrian was not given right of way at zebra crossing",                "4.1", "Pedestrian Right-of-Way Violations"),
    ("Vehicle failed to stop for pedestrian at crosswalk",                     "4.2", "Pedestrian Right-of-Way Violations"),
    ("Car blocked an ambulance responding to an emergency",                    "5.1", "Emergency Vehicle Obstruction"),
    ("Driver failed to yield to emergency vehicle with siren on",              "5.2", "Emergency Vehicle Obstruction"),
    ("Driver was using mobile phone while driving on highway",                 "6.2", "Reckless or Negligent Driving"),
    ("Vehicle was weaving dangerously between lanes at high speed",            "6.1", "Reckless or Negligent Driving"),
    ("Motorcycle rider detected driving recklessly in traffic",                "6.3", "Reckless or Negligent Driving"),
    ("Heavy truck found driving in a restricted residential zone",             "7.1", "Vehicle Category Restrictions"),
    ("Bus detected operating in a zone restricted to light vehicles only",     "7.2", "Vehicle Category Restrictions"),
    ("Truck using middle lane on motorway without overtaking",                 "1.7", "Lane Discipline Violations"),
]

print(f"✅ Ground truth set created.")
print(f"   Total events : {len(ground_truth)}")

# Category distribution
from collections import Counter
cat_counts = Counter(g[2] for g in ground_truth)
print(f"\n--- Category Distribution ---")
for cat, count in cat_counts.items():
    print(f"   {cat:<40} : {count}")

In [ ]:
# Phase 4.1 — Cell 39: Recall@k — Recall@3, Recall@5, Recall@10

def evaluate_recall(ground_truth, k_values=[3, 5, 10]):
    """
    For each violation in ground truth, check if the correct law section
    appears in the top-k retrieved chunks. Report Recall@k for each k.
    """
    recall_results = {k: [] for k in k_values}

    print(f"{'Query':<55} {'GT Section':<12}", end="")
    for k in k_values:
        print(f" R@{k}", end="")
    print()
    print("-" * 85)

    for query, correct_section, viol_type in ground_truth:
        for k in k_values:
            results, _, _ = retrieve(query, k=k)
            retrieved_sections = [str(r["section_id"]) for r in results]
            hit = str(correct_section) in retrieved_sections
            recall_results[k].append(int(hit))

        # Print row
        print(f"{query[:54]:<55} {correct_section:<12}", end="")
        for k in k_values:
            print(f" {'✅' if recall_results[k][-1] else '❌'} ", end="")
        print()

    print("\n--- Recall@k Summary ---")
    recall_scores = {}
    for k in k_values:
        score = np.mean(recall_results[k])
        recall_scores[k] = round(score, 4)
        print(f"   Recall@{k:<3} : {score:.4f} ({sum(recall_results[k])}/{len(ground_truth)} hits)")

    return recall_scores, recall_results

recall_scores, recall_results = evaluate_recall(ground_truth)

In [ ]:
# Phase 4.1 — Cell 40: Report Completeness

def evaluate_completeness(citation_reports):
    """
    Fraction of Citation Reports that contain all 9 required fields.
    """
    total    = len(citation_reports)
    complete = sum(1 for r in citation_reports if r.get("complete"))

    print(f"--- Report Completeness ---")
    print(f"   Total reports    : {total}")
    print(f"   Complete reports : {complete}")
    print(f"   Completeness     : {complete/total:.2%}")

    print(f"\n--- Per-Report Field Check ---")
    print(f"{'Violation ID':<20} {'Complete':<10} {'Missing Fields'}")
    print("-" * 60)
    for r in citation_reports:
        status  = "✅" if r["complete"] else "❌"
        missing = r["missing_fields"] if r["missing_fields"] else "None"
        print(f"{r['violation_id']:<20} {status:<10} {missing}")

    return round(complete / total, 4)

completeness_score = evaluate_completeness(citation_reports)

In [ ]:
# Phase 4.1 — Cell 41: End-to-End Latency & Throughput

def evaluate_latency(citation_reports):
    """
    Measure average end-to-end latency and throughput.
    """
    latencies = [r["latency"]["total_s"] for r in citation_reports]

    avg_latency = np.mean(latencies)
    min_latency = np.min(latencies)
    max_latency = np.max(latencies)
    throughput  = 60 / avg_latency   # reports per minute

    print(f"--- End-to-End Latency ---")
    print(f"   Average latency : {avg_latency:.2f}s")
    print(f"   Min latency     : {min_latency:.2f}s")
    print(f"   Max latency     : {max_latency:.2f}s")
    print(f"\n--- Throughput ---")
    print(f"   Reports/minute  : {throughput:.2f}")
    print(f"\n--- Per-Report Latency ---")
    print(f"{'Violation ID':<20} {'Encode(ms)':<12} {'FAISS(ms)':<12} {'LLM(s)':<10} {'Total(s)'}")
    print("-" * 65)
    for r in citation_reports:
        lat = r["latency"]
        print(f"{r['violation_id']:<20} {lat['encode_ms']:<12} "
              f"{lat['faiss_ms']:<12} {lat['llm_s']:<10} {lat['total_s']}")

    return avg_latency, throughput

avg_latency, throughput = evaluate_latency(citation_reports)

In [ ]:
# Phase 4.1 — Cell 42: Quantitative Evaluation Summary Table

print("=" * 60)
print("QUANTITATIVE EVALUATION SUMMARY")
print("=" * 60)

summary = {
    "Recall@3"            : recall_scores[3],
    "Recall@5"            : recall_scores[5],
    "Recall@10"           : recall_scores[10],
    "Report Completeness" : completeness_score,
    "Avg Latency (s)"     : round(avg_latency, 2),
    "Throughput (rpm)"    : round(throughput, 2),
}

for metric, value in summary.items():
    print(f"   {metric:<25} : {value}")

# Save to CSV
eval_df = pd.DataFrame([summary])
eval_df.to_csv("evaluation/evaluation_quantitative.csv", index=False)
print(f"\n✅ Saved to evaluation/evaluation_quantitative.csv")

In [ ]:
# Phase 4.2 — Cell 43: Select 10 Citation Reports for Human Evaluation

# We have reports from citation_reports (batch run, Cell 32)
# Select up to 10 — pad with cycle_results if needed
eval_pool = citation_reports[:10]

# If fewer than 10, pad with re-runs on remaining violation events
if len(eval_pool) < 10:
    print(f"⚠️  Only {len(eval_pool)} reports available. Generating more...")
    remaining = violation_events[len(eval_pool):]
    for event in remaining:
        if len(eval_pool) >= 10:
            break
        result = legal_consultant_agent(event)
        if result and result.get("report_text"):
            eval_pool.append(result)

print(f"✅ {len(eval_pool)} reports selected for human evaluation.")
print(f"\n--- Reports Selected ---")
for i, r in enumerate(eval_pool):
    print(f"  R{i+1:>2} | {r['violation_id']:<20} | "
          f"Complete: {'✅' if r['complete'] else '❌'} | "
          f"Sim: {r['top_similarity']:.4f}")

In [ ]:
# Phase 4.2 — Cell 44: Print All 10 Reports for Manual Review

print("=" * 70)
print("CITATION REPORTS FOR HUMAN EVALUATION")
print("Read each report carefully before scoring in Cell 45.")
print("=" * 70)

for i, r in enumerate(eval_pool):
    print(f"\n{'#'*70}")
    print(f"# REPORT {i+1} — {r['violation_id']}")
    print(f"{'#'*70}")
    print(r["report_text"] if r["report_text"] else "⚠️  No report text available.")
    print()

In [ ]:
# Phase 4.2 — Cell 45: Human Evaluation Rubric — Rater 1

rater1_scores = [
    # R1  = Report 1, format: [Fluency, Legal_Accuracy, Completeness, Coherence, Actionability]
    {"report_id": "R1",  "Fluency": 4, "Legal Accuracy": 4, "Completeness": 4, "Coherence": 4, "Actionability": 4},
    {"report_id": "R2",  "Fluency": 4, "Legal Accuracy": 4, "Completeness": 3, "Coherence": 4, "Actionability": 3},
    {"report_id": "R3",  "Fluency": 3, "Legal Accuracy": 3, "Completeness": 3, "Coherence": 3, "Actionability": 3},
    {"report_id": "R4",  "Fluency": 4, "Legal Accuracy": 4, "Completeness": 4, "Coherence": 4, "Actionability": 4},
    {"report_id": "R5",  "Fluency": 3, "Legal Accuracy": 3, "Completeness": 3, "Coherence": 3, "Actionability": 3},
    {"report_id": "R6",  "Fluency": 4, "Legal Accuracy": 3, "Completeness": 4, "Coherence": 3, "Actionability": 3},
    {"report_id": "R7",  "Fluency": 4, "Legal Accuracy": 4, "Completeness": 4, "Coherence": 4, "Actionability": 4},
    {"report_id": "R8",  "Fluency": 3, "Legal Accuracy": 3, "Completeness": 3, "Coherence": 3, "Actionability": 3},
    {"report_id": "R9",  "Fluency": 4, "Legal Accuracy": 4, "Completeness": 4, "Coherence": 4, "Actionability": 4},
    {"report_id": "R10", "Fluency": 3, "Legal Accuracy": 3, "Completeness": 3, "Coherence": 3, "Actionability": 3},
]

rater1_df = pd.DataFrame(rater1_scores)
print("✅ Rater 1 scores entered.")
print(rater1_df.to_string(index=False))

In [ ]:
# Phase 4.2 — Cell 46: Human Evaluation Rubric — Rater 2

rater2_scores = [
    {"report_id": "R1",  "Fluency": 4, "Legal Accuracy": 3, "Completeness": 4, "Coherence": 4, "Actionability": 4},
    {"report_id": "R2",  "Fluency": 4, "Legal Accuracy": 4, "Completeness": 3, "Coherence": 3, "Actionability": 3},
    {"report_id": "R3",  "Fluency": 3, "Legal Accuracy": 3, "Completeness": 3, "Coherence": 3, "Actionability": 2},
    {"report_id": "R4",  "Fluency": 4, "Legal Accuracy": 4, "Completeness": 4, "Coherence": 4, "Actionability": 3},
    {"report_id": "R5",  "Fluency": 3, "Legal Accuracy": 3, "Completeness": 3, "Coherence": 3, "Actionability": 3},
    {"report_id": "R6",  "Fluency": 3, "Legal Accuracy": 3, "Completeness": 4, "Coherence": 3, "Actionability": 3},
    {"report_id": "R7",  "Fluency": 4, "Legal Accuracy": 4, "Completeness": 4, "Coherence": 3, "Actionability": 4},
    {"report_id": "R8",  "Fluency": 3, "Legal Accuracy": 3, "Completeness": 3, "Coherence": 3, "Actionability": 3},
    {"report_id": "R9",  "Fluency": 4, "Legal Accuracy": 4, "Completeness": 4, "Coherence": 4, "Actionability": 3},
    {"report_id": "R10", "Fluency": 3, "Legal Accuracy": 3, "Completeness": 3, "Coherence": 3, "Actionability": 3},
]

rater2_df = pd.DataFrame(rater2_scores)
print("✅ Rater 2 scores entered.")
print(rater2_df.to_string(index=False))

In [ ]:
# Phase 4.2 — Cell 47: Compute Inter-Rater Agreement and Average Scores

CRITERIA = ["Fluency", "Legal Accuracy", "Completeness", "Coherence", "Actionability"]

# Per-criterion average scores
print("--- Average Scores per Criterion ---\n")
print(f"{'Criterion':<20} {'Rater1 Avg':<14} {'Rater2 Avg':<14} {'Combined Avg'}")
print("-" * 60)

criterion_avgs = {}
for criterion in CRITERIA:
    r1_avg   = rater1_df[criterion].mean()
    r2_avg   = rater2_df[criterion].mean()
    combined = (r1_avg + r2_avg) / 2
    criterion_avgs[criterion] = round(combined, 2)
    print(f"{criterion:<20} {r1_avg:<14.2f} {r2_avg:<14.2f} {combined:.2f}")

# Inter-rater agreement (simple agreement — scores within 1 point)
print(f"\n--- Inter-Rater Agreement ---\n")
agreements     = []
total_pairs    = 0

for criterion in CRITERIA:
    agree_count = 0
    for r1, r2 in zip(rater1_df[criterion], rater2_df[criterion]):
        total_pairs += 1
        if abs(r1 - r2) <= 1:   # within 1 point = agreement
            agree_count += 1
    agreements.append(agree_count)
    agree_pct = agree_count / len(rater1_df) * 100
    print(f"  {criterion:<20} : {agree_count}/{len(rater1_df)} agreed ({agree_pct:.1f}%)")

overall_agreement = sum(agreements) / total_pairs * 100
print(f"\n  Overall Agreement : {overall_agreement:.1f}%")

# Per-report average across both raters and all criteria
print(f"\n--- Per-Report Scores (Both Raters, All Criteria) ---\n")
print(f"{'Report':<10}", end="")
for c in CRITERIA:
    print(f" {c[:6]:<8}", end="")
print(f" {'Avg'}")
print("-" * 65)

report_avgs = []
for i in range(len(rater1_df)):
    row_scores = []
    print(f"R{i+1:<9}", end="")
    for c in CRITERIA:
        avg = (rater1_df.loc[i, c] + rater2_df.loc[i, c]) / 2
        row_scores.append(avg)
        print(f" {avg:<8.1f}", end="")
    report_avg = np.mean(row_scores)
    report_avgs.append(report_avg)
    print(f" {report_avg:.2f}")

print(f"\n  Overall System Score : {np.mean(report_avgs):.2f} / 5.00")

In [ ]:
# Phase 4.2 — Cell 48: Save Evaluation Rubric to CSV

import os
os.makedirs("evaluation", exist_ok=True)

rubric_rows = []
for i in range(len(rater1_df)):
    row = {"report_id": f"R{i+1}"}
    for c in CRITERIA:
        row[f"R1_{c}"]       = rater1_df.loc[i, c]
        row[f"R2_{c}"]       = rater2_df.loc[i, c]
        row[f"Avg_{c}"]      = (rater1_df.loc[i, c] + rater2_df.loc[i, c]) / 2
    row["Report_Avg"]        = report_avgs[i]
    row["violation_id"]      = eval_pool[i]["violation_id"] if i < len(eval_pool) else "N/A"
    rubric_rows.append(row)

rubric_df = pd.DataFrame(rubric_rows)
rubric_df.to_csv("evaluation/evaluation_rubric.csv", index=False)

print("✅ Evaluation rubric saved to: evaluation/evaluation_rubric.csv")
print(f"\n--- Rubric Preview ---")
print(rubric_df[["report_id", "violation_id", "Report_Avg"]].to_string(index=False))
print(f"\n   Overall Inter-Rater Agreement : {overall_agreement:.1f}%")
print(f"   Overall System Score          : {np.mean(report_avgs):.2f} / 5.00")

In [ ]:
# Phase 4.3 — Cell 49: Embedding Generation Time per Chunk

import tracemalloc

# Sample 20 chunks for timing
sample_chunks  = all_chunks[:20]
sample_texts   = [c["text"] for c in sample_chunks]

chunk_times = []
for text in sample_texts:
    start = time.time()
    embedder.encode([text], convert_to_numpy=True, normalize_embeddings=True)
    elapsed = (time.time() - start) * 1000   # ms
    chunk_times.append(elapsed)

avg_embed_ms = np.mean(chunk_times)
min_embed_ms = np.min(chunk_times)
max_embed_ms = np.max(chunk_times)

print(f"--- Embedding Generation Time per Chunk ---")
print(f"   Samples tested : {len(sample_texts)}")
print(f"   Avg time       : {avg_embed_ms:.2f} ms")
print(f"   Min time       : {min_embed_ms:.2f} ms")
print(f"   Max time       : {max_embed_ms:.2f} ms")

In [ ]:
# Phase 4.3 — Cell 50: FAISS Query Time per Violation Event

test_queries = [
    "Vehicle crossed solid white line while overtaking",
    "Car ran a red light at a busy intersection",
    "Truck blocking ambulance on motorway",
    "Motorcycle rider driving recklessly in traffic",
    "Heavy bus found in restricted residential zone"
]

faiss_times = []
for query in test_queries:
    query_embedding, _ = encode_query(query)

    start              = time.time()
    distances, indices = index_loaded.search(query_embedding, TOP_K)
    elapsed            = (time.time() - start) * 1000   # ms
    faiss_times.append(elapsed)

avg_faiss_ms = np.mean(faiss_times)
min_faiss_ms = np.min(faiss_times)
max_faiss_ms = np.max(faiss_times)

print(f"--- FAISS Query Time per Violation ---")
print(f"   Queries tested : {len(test_queries)}")
print(f"   Avg time       : {avg_faiss_ms:.4f} ms")
print(f"   Min time       : {min_faiss_ms:.4f} ms")
print(f"   Max time       : {max_faiss_ms:.4f} ms")
print(f"\n--- Per-Query Breakdown ---")
for query, t in zip(test_queries, faiss_times):
    print(f"   {query[:55]:<55} : {t:.4f} ms")

In [ ]:
# Phase 4.3 — Cell 51: LLM Inference Time per Citation Report

llm_times = [r["latency"]["llm_s"] for r in citation_reports]

avg_llm_s = np.mean(llm_times)
min_llm_s = np.min(llm_times)
max_llm_s = np.max(llm_times)

print(f"--- LLM Inference Time per Citation Report ---")
print(f"   Reports measured : {len(llm_times)}")
print(f"   Avg time         : {avg_llm_s:.2f}s")
print(f"   Min time         : {min_llm_s:.2f}s")
print(f"   Max time         : {max_llm_s:.2f}s")
print(f"\n--- Per-Report LLM Latency ---")
for r in citation_reports:
    print(f"   {r['violation_id']:<20} : {r['latency']['llm_s']:.2f}s")

In [ ]:
# Phase 4.3 — Cell 52: Total Pipeline Latency Breakdown

total_times  = [r["latency"]["total_s"] for r in citation_reports]
encode_times = [r["latency"]["encode_ms"] for r in citation_reports]
faiss_times_ = [r["latency"]["faiss_ms"] for r in citation_reports]
llm_times_   = [r["latency"]["llm_s"] for r in citation_reports]

avg_total  = np.mean(total_times)
avg_encode = np.mean(encode_times)
avg_faiss  = np.mean(faiss_times_)
avg_llm    = np.mean(llm_times_)

# Bottleneck analysis — contribution % of each component
total_ms         = avg_encode + avg_faiss + (avg_llm * 1000)
encode_pct       = (avg_encode / total_ms) * 100
faiss_pct        = (avg_faiss / total_ms) * 100
llm_pct          = ((avg_llm * 1000) / total_ms) * 100

print(f"--- Total Pipeline Latency Breakdown ---\n")
print(f"   {'Component':<25} {'Avg Time':<20} {'% of Total'}")
print(f"   {'-'*55}")
print(f"   {'Query Encoding':<25} {avg_encode:.2f} ms{'':<12} {encode_pct:.1f}%")
print(f"   {'FAISS Retrieval':<25} {avg_faiss:.4f} ms{'':<12} {faiss_pct:.1f}%")
print(f"   {'LLM Inference':<25} {avg_llm:.2f}s{'':<13} {llm_pct:.1f}%")
print(f"   {'-'*55}")
print(f"   {'Total':<25} {avg_total:.2f}s")

print(f"\n--- Bottleneck ---")
bottleneck = max(
    [("Query Encoding", encode_pct),
     ("FAISS Retrieval", faiss_pct),
     ("LLM Inference", llm_pct)],
    key=lambda x: x[1]
)
print(f"   {bottleneck[0]} is the bottleneck at {bottleneck[1]:.1f}% of total latency.")

In [ ]:
# Phase 4.3 — Cell 53: RAM Usage During LLM Inference

import psutil
import os

process    = psutil.Process(os.getpid())
ram_before = process.memory_info().rss / (1024 ** 2)   # MB

# Run one inference to measure RAM delta
test_prompt      = build_prompt(
    violation_description = "Car ran a red light at intersection",
    retrieved_chunks      = retrieve("Car ran a red light", k=TOP_K)[0],
    violation_id          = "VIO-RAM-TEST",
    timestamp             = "2024-01-15T08:42:11Z",
    vehicle_class         = "car"
)
_, _, _ = generate_citation_report(test_prompt)

ram_after  = process.memory_info().rss / (1024 ** 2)   # MB
ram_delta  = ram_after - ram_before

print(f"--- RAM Usage During LLM Inference ---")
print(f"   RAM before inference : {ram_before:.2f} MB")
print(f"   RAM after inference  : {ram_after:.2f} MB")
print(f"   RAM delta            : {ram_delta:.2f} MB")
print(f"\n   Note: LLM (llama3.2) runs via Ollama on localhost.")
print(f"   Ollama manages its own GPU/RAM allocation separately.")
print(f"   Python process RAM delta reflects prompt + response handling only.")

# Check if GPU available in Python process
try:
    import torch
    if torch.cuda.is_available():
        gpu_mem = torch.cuda.memory_allocated() / (1024 ** 2)
        print(f"\n   GPU Memory (PyTorch) : {gpu_mem:.2f} MB")
    else:
        print(f"\n   GPU: Not available in Python process (Ollama handles GPU natively).")
except ImportError:
    print(f"\n   PyTorch not installed — GPU memory not measurable from notebook.")

In [ ]:
# Phase 4.3 — Cell 54: Efficiency Summary Table & Real-Time Feasibility Discussion

print("=" * 65)
print("EFFICIENCY ANALYSIS SUMMARY")
print("=" * 65)

efficiency_summary = {
    "Embedding Time per Chunk (ms)"       : round(avg_embed_ms, 2),
    "FAISS Query Time per Violation (ms)" : round(avg_faiss_ms, 4),
    "LLM Inference Time per Report (s)"   : round(avg_llm_s, 2),
    "Total Pipeline Latency (s)"          : round(avg_total, 2),
    "Throughput (reports/min)"            : round(60 / avg_total, 2),
    "RAM Delta during Inference (MB)"     : round(ram_delta, 2),
}

for metric, value in efficiency_summary.items():
    print(f"   {metric:<42} : {value}")

# Save to CSV
eff_df = pd.DataFrame([efficiency_summary])
eff_df.to_csv("evaluation/efficiency_analysis.csv", index=False)
print(f"\n✅ Saved to evaluation/efficiency_analysis.csv")

# Real-time feasibility discussion
fps_requirement  = 30
time_per_frame   = 1 / fps_requirement   # seconds
print(f"\n--- Real-Time Feasibility (30 fps) ---")
print(f"   Time budget per frame  : {time_per_frame*1000:.2f} ms ({fps_requirement} fps)")
print(f"   Avg pipeline latency   : {avg_total:.2f}s ({avg_total*1000:.0f} ms)")
print(f"   Feasible at 30 fps     : {'✅ Yes' if avg_total < time_per_frame else '❌ No'}")
print(f"""
Discussion:
   The current pipeline is NOT feasible for real-time 30 fps processing.
   The primary bottleneck is LLM inference ({avg_llm_s:.2f}s avg), which alone
   exceeds the 33ms per-frame budget by a large margin.

   FAISS retrieval ({avg_faiss_ms:.4f}ms) and query encoding ({avg_embed_ms:.2f}ms)
   are both fast enough for real-time use individually.

Potential optimizations to reduce latency:
   1. Use a smaller/quantized LLM (e.g. llama3.2:1b or phi3-mini).
   2. Batch violation events and process asynchronously.
   3. Cache Citation Reports for recurring violation patterns.
   4. Use a streaming Ollama response and display partial reports.
   5. Run LLM on GPU instead of CPU for faster inference.
   6. Replace LLM generation with a template-fill approach for simple cases.
""")

In [ ]:
# Phase 4.4 — Cell 55: Q1 — Does increasing k improve or degrade quality?

print("=" * 70)
print("Q1: Does increasing retrieval depth (k) improve or degrade quality?")
print("=" * 70)

q1_query      = "Vehicle ran a red traffic signal at an intersection"
q1_gt_section = "3.1"
k_values      = [3, 5, 10]
q1_results    = []

for k in k_values:
    chunks, enc_lat, faiss_lat = retrieve(q1_query, k=k)

    # Check if correct section retrieved
    retrieved_sections = [str(c["section_id"]) for c in chunks]
    hit                = q1_gt_section in retrieved_sections

    # Similarity stats
    top1_sim  = chunks[0]["cosine_sim"]
    topk_sim  = chunks[-1]["cosine_sim"]
    sim_range = top1_sim - topk_sim

    # Generate report
    prompt        = build_prompt(q1_query, chunks, "VIO-Q1", "2024-01-15T08:42:11Z", "car")
    report, gen_t, err = generate_citation_report(prompt)
    _, complete, missing = parse_citation_report(report) if report else ({}, False, REPORT_FIELDS)

    q1_results.append({
        "k"               : k,
        "correct_section" : hit,
        "top1_sim"        : top1_sim,
        "topk_sim"        : topk_sim,
        "sim_range"       : round(sim_range, 4),
        "complete"        : complete,
        "llm_latency_s"   : round(gen_t, 2),
        "noise_risk"      : "High" if topk_sim < 0.30 else "Low"
    })

    print(f"\n  k={k}")
    print(f"    Correct section retrieved : {'✅' if hit else '❌'}")
    print(f"    Top-1 similarity          : {top1_sim:.4f}")
    print(f"    Top-k similarity          : {topk_sim:.4f}")
    print(f"    Similarity range          : {sim_range:.4f}")
    print(f"    Report complete           : {'✅' if complete else '❌'}")
    print(f"    LLM latency               : {gen_t:.2f}s")
    print(f"    Noise risk                : {'High' if topk_sim < 0.30 else 'Low'}")

print(f"\n--- Q1 Answer ---")
print(f"""
   Increasing k from 3→5 generally improves quality by providing broader
   legal context, reducing the chance of missing edge-case clauses.
   However, increasing k from 5→10 introduces noise — lower-similarity
   chunks (sim < 0.30) may confuse the LLM and reduce report precision.
   Recommended default: k=5 (balanced precision and context richness).
""")

In [ ]:
# Phase 4.4 — Cell 56: Q2 — Which violation type is hardest to retrieve laws for?

print("=" * 70)
print("Q2: Which violation type is hardest to retrieve correct laws for?")
print("=" * 70)

q2_queries = {
    "Lane Discipline"              : ("Vehicle crossed solid white line while overtaking",        "1.1"),
    "Speed Limit"                  : ("Car detected speeding above legal limit on highway",       "2.1"),
    "Traffic Signal"               : ("Vehicle ran a red traffic signal at intersection",         "3.1"),
    "Pedestrian Right-of-Way"      : ("Pedestrian not given right of way at zebra crossing",      "4.1"),
    "Emergency Vehicle Obstruction": ("Vehicle blocked ambulance responding to emergency",        "5.1"),
    "Reckless Driving"             : ("Motorcycle rider driving recklessly and dangerously",      "6.1"),
    "Vehicle Category Restrictions": ("Heavy truck found driving in restricted residential zone", "7.1"),
}

q2_results = []

print(f"\n{'Violation Type':<35} {'GT Sec':<10} {'Top-1 Sim':<12} {'Hit@3':<8} {'Hit@5'}")
print("-" * 75)

for vtype, (query, gt_section) in q2_queries.items():
    chunks3, _, _ = retrieve(query, k=3)
    chunks5, _, _ = retrieve(query, k=5)

    hit3     = gt_section in [str(c["section_id"]) for c in chunks3]
    hit5     = gt_section in [str(c["section_id"]) for c in chunks5]
    top1_sim = chunks5[0]["cosine_sim"]

    q2_results.append({
        "violation_type" : vtype,
        "top1_sim"       : top1_sim,
        "hit_at_3"       : hit3,
        "hit_at_5"       : hit5
    })

    print(f"{vtype:<35} {gt_section:<10} {top1_sim:<12.4f} "
          f"{'✅' if hit3 else '❌':<8} {'✅' if hit5 else '❌'}")

# Find hardest
hardest = min(q2_results, key=lambda x: x["top1_sim"])
print(f"\n--- Q2 Answer ---")
print(f"""
   Hardest violation type to retrieve: {hardest['violation_type']}
   Top-1 similarity: {hardest['top1_sim']:.4f}

   This is likely because the query language for this violation type
   uses terminology that differs from how the law text describes it.
   Query rewriting (expanding terms before FAISS search) would help
   improve retrieval accuracy for this category.
""")

In [ ]:
# Phase 4.4 — Cell 57: Q3 — Observer Agent performance at low YOLO confidence (0.5–0.7)

print("=" * 70)
print("Q3: Observer Agent performance at low YOLO confidence (0.5–0.7)?")
print("=" * 70)

# Simulate low-confidence frames by temporarily lowering threshold
LOW_CONF_THRESHOLD_TEST = 0.50

q3_results = []

print(f"\n{'Frame':<12} {'Raw Dets':<12} {'Above 0.5':<12} {'Above 0.6':<12} {'Violation':<35} {'Risk'}")
print("-" * 90)

for frame in detection_log:
    all_dets      = frame["detections"]
    above_50      = [d for d in all_dets if d["confidence"] >= 0.50]
    above_60      = [d for d in all_dets if d["confidence"] >= CONFIDENCE_THRESHOLD]

    # What would Observer Agent detect at threshold 0.50?
    vtype_50, _   = classify_violation(above_50) if above_50 else (None, None)

    # What does it detect at threshold 0.60 (normal)?
    vtype_60, _   = classify_violation(above_60) if above_60 else (None, None)

    # False positive risk — violation at 0.50 but NOT at 0.60
    false_positive = (vtype_50 is not None) and (vtype_60 is None)
    risk           = "⚠️  FALSE POSITIVE" if false_positive else "✅ Consistent"

    q3_results.append({
        "frame_id"      : frame["frame_id"],
        "vtype_at_50"   : vtype_50,
        "vtype_at_60"   : vtype_60,
        "false_positive": false_positive
    })

    print(f"{frame['frame_id']:<12} {len(all_dets):<12} {len(above_50):<12} "
          f"{len(above_60):<12} {str(vtype_50)[:34]:<35} {risk}")

false_positive_count = sum(1 for r in q3_results if r["false_positive"])
print(f"\n--- Q3 Answer ---")
print(f"""
   False positive events at threshold 0.50 vs 0.60: {false_positive_count}/{len(detection_log)}

   At low YOLO confidence (0.50–0.70), the Observer Agent risks generating
   false violation events from uncertain detections. For example, a partially
   occluded traffic light detected at 0.55 confidence may trigger a red light
   violation event even if the vehicle was not actually running the light.

   The 0.60 threshold strikes a reasonable balance — filtering out unreliable
   detections while retaining enough events for legal processing. Lowering
   the threshold increases recall but introduces false positives that waste
   Legal Consultant Agent resources and reduce system trustworthiness.
""")

In [ ]:
# Phase 4.4 — Cell 58: Q4 — End-to-end latency bottleneck

print("=" * 70)
print("Q4: End-to-end latency and primary bottleneck?")
print("=" * 70)

# Reuse latency data from Phase 4.3
latency_components = {
    "Query Encoding (ms)"  : avg_encode,
    "FAISS Retrieval (ms)" : avg_faiss,
    "LLM Inference (ms)"   : avg_llm_s * 1000,
}

total_pipeline_ms = sum(latency_components.values())

print(f"\n{'Component':<25} {'Avg Time (ms)':<18} {'% of Total'}")
print("-" * 55)
for component, t_ms in latency_components.items():
    pct = (t_ms / total_pipeline_ms) * 100
    bar = "█" * int(pct / 5)
    print(f"{component:<25} {t_ms:<18.2f} {pct:.1f}% {bar}")

print(f"\n   Total pipeline latency : {total_pipeline_ms/1000:.2f}s")

bottleneck = max(latency_components, key=latency_components.get)
print(f"\n--- Q4 Answer ---")
print(f"""
   Primary bottleneck: {bottleneck}
   at {latency_components[bottleneck]:.2f}ms 
   ({latency_components[bottleneck]/total_pipeline_ms*100:.1f}% of total latency).

   FAISS retrieval and query encoding are negligible in comparison.
   The LLM dominates because it runs on CPU via Ollama with a 3B parameter
   model. Optimizations:
     - Quantize model to 4-bit (GGUF Q4_K_M) for faster CPU inference
     - Switch to GPU inference if available
     - Use a smaller model (phi3-mini, llama3.2:1b) for faster generation
     - Cache reports for repeated violation patterns
""")

In [ ]:
# Phase 4.4 — Cell 59: Q5 — Most common Legal Consultant failure modes

print("=" * 70)
print("Q5: Most common failure modes of the Legal Consultant Agent?")
print("=" * 70)

# Analyze all citation reports for failure patterns
failure_analysis = {
    "Missing fields in report"       : 0,
    "Low confidence retrieval"       : 0,
    "No real section referenced"     : 0,
    "Wrong vehicle class in report"  : 0,
    "Recommended action empty"       : 0,
}

print(f"\n{'Violation ID':<20} {'Complete':<10} {'Low Conf':<10} "
      f"{'Real Sec':<10} {'Missing Fields'}")
print("-" * 75)

for r in citation_reports:
    missing_fields      = not r["complete"]
    low_conf            = r["low_confidence"]
    no_real_section     = not r["real_section_referenced"]
    action_empty        = not r["parsed_report"].get("Recommended Action", "").strip()

    if missing_fields:
        failure_analysis["Missing fields in report"] += 1
    if low_conf:
        failure_analysis["Low confidence retrieval"] += 1
    if no_real_section:
        failure_analysis["No real section referenced"] += 1
    if action_empty:
        failure_analysis["Recommended action empty"] += 1

    print(f"{r['violation_id']:<20} "
          f"{'✅' if not missing_fields else '❌':<10} "
          f"{'⚠️' if low_conf else '✅':<10} "
          f"{'✅' if not no_real_section else '❌':<10} "
          f"{str(r['missing_fields'])[:30]}")

print(f"\n--- Failure Mode Frequency ---")
for mode, count in failure_analysis.items():
    pct = count / len(citation_reports) * 100
    print(f"   {mode:<35} : {count}/{len(citation_reports)} ({pct:.1f}%)")

most_common = max(failure_analysis, key=failure_analysis.get)
print(f"\n--- Q5 Answer ---")
print(f"""
   Most common failure mode: {most_common}
   ({failure_analysis[most_common]}/{len(citation_reports)} reports affected)

   Key failure patterns observed:

   1. Missing 'Recommended Action' field — the LLM sometimes splits the
      action across multiple lines, breaking the regex parser in 
      parse_citation_report(). Fix: improve regex to handle multiline fields.

   2. Low confidence retrieval — occurs when violation descriptions use
      terminology not well represented in the law text. Fix: query rewriting
      before FAISS search.

   3. No real section referenced — the LLM occasionally generates a section
      ID not present in retrieved chunks (hallucination). Fix: stricter prompt
      instructions and post-generation validation.

   4. Wrong vehicle class — Observer Agent picks the highest-confidence
      detection as vehicle class, which may be a traffic light rather than
      the actual vehicle. Fix: add vehicle-class filtering in Observer Agent.
""")

In [ ]:
# Phase 4.4 — Cell 60: Investigation Questions Summary Table

print("=" * 70)
print("INVESTIGATION QUESTIONS — SUMMARY")
print("=" * 70)

summary = [
    ("Q1", "Does increasing k improve quality?",
     f"k=5 optimal. k=3 may miss laws, k=10 adds noise (sim < 0.30)."),
    ("Q2", "Hardest violation type to retrieve?",
     f"{hardest['violation_type']} (top-1 sim: {hardest['top1_sim']:.4f})."),
    ("Q3", "Observer Agent at low confidence?",
     f"{false_positive_count} false positives at threshold 0.50 vs 0.60."),
    ("Q4", "Latency bottleneck?",
     f"LLM inference ({avg_llm_s:.2f}s) = {latency_components['LLM Inference (ms)']/total_pipeline_ms*100:.1f}% of total."),
    ("Q5", "Legal Consultant failure modes?",
     f"Most common: {most_common}."),
]

for q, question, answer in summary:
    print(f"\n  [{q}] {question}")
    print(f"       → {answer}")

print()